# Test Retrieval from Milvus and SQLite Databases

This notebook tests:
1. Vector similarity search from Milvus Lite database
2. Reranking results using a cross-encoder
3. Querying structured data from SQLite3 database
4. Combining results from both databases

## 1. Setup and Import Dependencies

In [1]:
import sqlite3
from langchain_milvus import Milvus
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

print("All dependencies imported successfully!")

All dependencies imported successfully!


## 2. Load Milvus Vector Store and Setup Reranker

In [2]:
# Initialize embeddings (same as used during data generation)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load existing Milvus vector store
URI = "../../data/parking.db"
vector_store = Milvus(
    embedding_function=embeddings,
    connection_args={"uri": URI},
    collection_name="parking_policy_collection",
    drop_old=False  # Load existing data
)

print(f"✓ Milvus vector store loaded from {URI}")

/home/tamer/personal-projects/Chatbot-for-Parking-Space-Reservation/.venv/lib/python3.11/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


✓ Milvus vector store loaded from ../../data/parking.db


In [3]:
# Setup reranker pipeline
print("Setting up reranker...")

# Base retriever fetches top-k candidates
base_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

# Cross-encoder reranker model
reranker_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

# Compressor that reranks and keeps top-n
compressor = CrossEncoderReranker(model=reranker_model, top_n=3)

# Final retrieval pipeline with reranking
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print("✓ Reranker setup complete!")

Setting up reranker...
✓ Reranker setup complete!


## 3. Connect to SQLite Database

In [4]:
# Connect to SQLite database
sqlite_db_path = "../../data/parking_db.sqlite"
conn = sqlite3.connect(sqlite_db_path)
cursor = conn.cursor()

print(f"✓ Connected to SQLite database: {sqlite_db_path}")

# Verify tables exist
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(f"\nAvailable tables: {[table[0] for table in tables]}")

✓ Connected to SQLite database: ../../data/parking_db.sqlite

Available tables: ['parking_spots', 'sqlite_sequence', 'reservations']


## 4. Test Vector Retrieval with Reranking

In [5]:
# Test 1: Basic similarity search (without reranking)
test_query_1 = "What are the operating hours?"

print(f"\n{'='*60}")
print(f"TEST 1: Basic Similarity Search")
print(f"Query: '{test_query_1}'")
print(f"{'='*60}\n")

basic_results = vector_store.similarity_search(test_query_1, k=3)

for i, doc in enumerate(basic_results, 1):
    print(f"--- Result {i} (Basic) ---")
    print(doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content)
    print()


TEST 1: Basic Similarity Search
Query: 'What are the operating hours?'

--- Result 1 (Basic) ---
## 2. Operating Hours & Access

- **Standard Hours:** The facility is open 24 hours a day, 7 days a week.
- **Staffed Hours:** On-site customer support is available from 8:00 AM to 8:00 PM daily.
- **...

--- Result 2 (Basic) ---
- **Grace Period:** We offer a 15-minute grace period before and after your reservation time.

--- Result 3 (Basic) ---
## 3. Pricing Policy (Base Rates)

_Note: Prices are subject to dynamic adjustments based on demand. Check real-time availability for exact quotes._

- **Hourly Rate:** $5.00 per hour (or part thereof...



In [6]:
# Test 2: Retrieval with Reranking
test_query_2 = "Are heavy commercial trucks allowed?"

print(f"\n{'='*60}")
print(f"TEST 2: Retrieval with Reranking")
print(f"Query: '{test_query_2}'")
print(f"{'='*60}\n")

reranked_results = compression_retriever.invoke(test_query_2)

for i, doc in enumerate(reranked_results, 1):
    print(f"--- Reranked Result {i} ---")
    print(doc.page_content)
    print()


TEST 2: Retrieval with Reranking
Query: 'Are heavy commercial trucks allowed?'

--- Reranked Result 1 ---
- **Address:** 101 Innovation Blvd, Tech District, Metro City, 54321.
- **Entrance:** Main entrance is located on the North side of Innovation Blvd, accessible via the right-hand lane.
- **Height Clearance:** Maximum vehicle height is 2.1 meters (6.8 feet).
- **Vehicle Types:** We accept sedans, SUVs, and motorcycles. Heavy commercial trucks, trailers, and RVs are strictly prohibited.

## 2. Operating Hours & Access

--- Reranked Result 2 ---
## 6. Rules of Conduct

- **Speed Limit:** 10 km/h (6 mph) inside the facility.
- **Parking Discipline:** Vehicles must be parked within the marked lines. Taking up two spots may result in towing at the owner's expense.
- **Prohibited Items:** Storage of flammable materials or leaving unattended baggage is strictly prohibited.

--- Reranked Result 3 ---
## 5. Amenities & Services

- **EV Charging:** Level 2 Electric Vehicle chargers are avail

In [7]:
# Test 3: Query about pricing
test_query_3 = "How much does parking cost per hour?"

print(f"\n{'='*60}")
print(f"TEST 3: Pricing Query with Reranking")
print(f"Query: '{test_query_3}'")
print(f"{'='*60}\n")

reranked_results = compression_retriever.invoke(test_query_3)

for i, doc in enumerate(reranked_results, 1):
    print(f"--- Reranked Result {i} ---")
    print(doc.page_content)
    print()


TEST 3: Pricing Query with Reranking
Query: 'How much does parking cost per hour?'

--- Reranked Result 1 ---
## 3. Pricing Policy (Base Rates)

_Note: Prices are subject to dynamic adjustments based on demand. Check real-time availability for exact quotes._

- **Hourly Rate:** $5.00 per hour (or part thereof).
- **Daily Max:** $35.00 for any 24-hour period.
- **Overnight Flat Rate:** $15.00 (Entry after 6:00 PM, Exit before 8:00 AM).
- **Lost Ticket/Reservation:** A flat fee of $50.00 applies if validation cannot be confirmed.

## 4. Reservation & Booking Process

--- Reranked Result 2 ---
## 6. Rules of Conduct

- **Speed Limit:** 10 km/h (6 mph) inside the facility.
- **Parking Discipline:** Vehicles must be parked within the marked lines. Taking up two spots may result in towing at the owner's expense.
- **Prohibited Items:** Storage of flammable materials or leaving unattended baggage is strictly prohibited.

--- Reranked Result 3 ---
## 2. Operating Hours & Access

- **Standard 

## 5. Test SQLite Database Queries

In [ ]:
# Test 4: Query available parking spots
print(f"\n{'='*60}")
print(f"TEST 4: Available Parking Spots")
print(f"{'='*60}\n")

cursor.execute("""
    SELECT spot_number, spot_type, floor, status, price_per_hour
    FROM parking_spots
    WHERE status = 'available'
""")

available_spots = cursor.fetchall()

print(f"Found {len(available_spots)} available spots:\n")
for spot in available_spots:
    print(f"  • {spot[0]} - {spot[1]} ({spot[2]}) - ${spot[4]}/hour")


TEST 4: Available Parking Spots

Found 8 available spots:

  • G-01 - Accessible (Ground) - $5.0/hour
  • G-02 - Accessible (Ground) - $5.0/hour
  • G-03 - Standard (Ground) - $5.0/hour
  • L1-A1 - Standard (Level 1) - $5.0/hour
  • L1-A2 - Standard (Level 1) - $5.0/hour
  • L2-A1 - EV (Level 2) - $5.0/hour
  • L2-A2 - EV (Level 2) - $5.0/hour
  • L2-B1 - Standard (Level 2) - $5.0/hour


I0000 00:00:1772520246.058739   10328 chttp2_transport.cc:1353] unix:/tmp/tmpx7_79m8x_parking.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11}
E0000 00:00:1772520246.058895   10328 chttp2_transport.cc:1385] unix:/tmp/tmpx7_79m8x_parking.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


In [9]:
# Test 5: Query EV charging spots
print(f"\n{'='*60}")
print(f"TEST 5: EV Charging Spots")
print(f"{'='*60}\n")

cursor.execute("""
    SELECT spot_number, floor, status
    FROM parking_spots
    WHERE spot_type = 'EV'
""")

ev_spots = cursor.fetchall()

print(f"Found {len(ev_spots)} EV charging spots:\n")
for spot in ev_spots:
    print(f"  • {spot[0]} on {spot[1]} - Status: {spot[2]}")


TEST 5: EV Charging Spots

Found 2 EV charging spots:

  • L2-A1 on Level 2 - Status: available
  • L2-A2 on Level 2 - Status: available


In [10]:
# Test 6: Query accessible spots
print(f"\n{'='*60}")
print(f"TEST 6: Accessible Parking Spots")
print(f"{'='*60}\n")

cursor.execute("""
    SELECT spot_number, floor, status
    FROM parking_spots
    WHERE spot_type = 'Accessible'
""")

accessible_spots = cursor.fetchall()

print(f"Found {len(accessible_spots)} accessible spots:\n")
for spot in accessible_spots:
    print(f"  • {spot[0]} on {spot[1]} - Status: {spot[2]}")


TEST 6: Accessible Parking Spots

Found 2 accessible spots:

  • G-01 on Ground - Status: available
  • G-02 on Ground - Status: available


## 6. Combined Test: Policy + Database Query

In [11]:
# Test 7: Combined retrieval - user asks about EV charging
user_query = "Where can I find EV charging stations and are they available?"

print(f"\n{'='*60}")
print(f"TEST 7: Combined Query (Vector + SQL)")
print(f"User Query: '{user_query}'")
print(f"{'='*60}\n")

# Step 1: Get policy information from vector store
print("[1] Retrieving policy information from vector store...\n")
policy_docs = compression_retriever.invoke(user_query)

print("Policy Information:")
for i, doc in enumerate(policy_docs, 1):
    print(f"\n--- Policy Excerpt {i} ---")
    print(doc.page_content[:300] + "..." if len(doc.page_content) > 300 else doc.page_content)

# Step 2: Get real-time availability from SQLite
print("\n" + "="*60)
print("[2] Checking real-time EV spot availability in database...\n")

cursor.execute("""
    SELECT spot_number, floor, status, price_per_hour
    FROM parking_spots
    WHERE spot_type = 'EV'
""")

ev_spots = cursor.fetchall()

print("Real-time EV Spot Status:")
for spot in ev_spots:
    status_emoji = "✅" if spot[2] == 'available' else "❌"
    print(f"  {status_emoji} {spot[0]} on {spot[1]} - {spot[2].upper()} - ${spot[3]}/hour")

print("\n" + "="*60)
print("Combined Answer: Based on policy and real-time data")
print("="*60)


TEST 7: Combined Query (Vector + SQL)
User Query: 'Where can I find EV charging stations and are they available?'

[1] Retrieving policy information from vector store...

Policy Information:

--- Policy Excerpt 1 ---
## 5. Amenities & Services

- **EV Charging:** Level 2 Electric Vehicle chargers are available on **Level 2, Row A**. Standard parking rates apply while charging.
- **Accessibility:** Designated accessible spots are located on the **Ground Floor** near the elevator banks.
- **Security:** The facilit...

--- Policy Excerpt 2 ---
## 2. Operating Hours & Access

- **Standard Hours:** The facility is open 24 hours a day, 7 days a week.
- **Staffed Hours:** On-site customer support is available from 8:00 AM to 8:00 PM daily.
- **After-Hours Access:** Pedestrian access to the facility after 10:00 PM requires scanning a valid act...

--- Policy Excerpt 3 ---
- **Address:** 101 Innovation Blvd, Tech District, Metro City, 54321.
- **Entrance:** Main entrance is located on the Nort

## 7. Database Statistics

In [12]:
# Test 8: Overall statistics
print(f"\n{'='*60}")
print(f"DATABASE STATISTICS")
print(f"{'='*60}\n")

# Total spots
cursor.execute("SELECT COUNT(*) FROM parking_spots")
total_spots = cursor.fetchone()[0]
print(f"Total parking spots: {total_spots}")

# Available spots
cursor.execute("SELECT COUNT(*) FROM parking_spots WHERE status = 'available'")
available_count = cursor.fetchone()[0]
print(f"Available spots: {available_count}")

# By type
cursor.execute("""
    SELECT spot_type, COUNT(*), 
           SUM(CASE WHEN status = 'available' THEN 1 ELSE 0 END) as available
    FROM parking_spots
    GROUP BY spot_type
""")

type_stats = cursor.fetchall()
print("\nBreakdown by type:")
for stat in type_stats:
    print(f"  • {stat[0]}: {stat[2]}/{stat[1]} available")

# By floor
cursor.execute("""
    SELECT floor, COUNT(*),
           SUM(CASE WHEN status = 'available' THEN 1 ELSE 0 END) as available
    FROM parking_spots
    GROUP BY floor
""")

floor_stats = cursor.fetchall()
print("\nBreakdown by floor:")
for stat in floor_stats:
    print(f"  • {stat[0]}: {stat[2]}/{stat[1]} available")


DATABASE STATISTICS

Total parking spots: 9
Available spots: 8

Breakdown by type:
  • Accessible: 2/2 available
  • EV: 2/2 available
  • Standard: 4/5 available

Breakdown by floor:
  • Ground: 3/3 available
  • Level 1: 2/3 available
  • Level 2: 3/3 available


## 8. Cleanup

In [13]:
# Close SQLite connection
conn.close()
print("\n✓ SQLite connection closed.")
print("\n" + "="*60)
print("All tests completed successfully!")
print("="*60)


✓ SQLite connection closed.

All tests completed successfully!
